# Data Quality Report (Polars)

Flags implausible delays, stale observations, pre-trip/post-trip observations, and VM vs stop-call delay disagreement before delay metrics are computed using the Polars analysis path.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

quality = load_polars_script("polars_data_quality_report", "data-quality-report.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
LIMIT = 20
MIN_OBSERVATIONS = 50
TIMEZONE = "Europe/Helsinki"


Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [ ]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    limit = LIMIT
    min_observations = MIN_OBSERVATIONS
    timezone = TIMEZONE

rows = quality.load_quality_rows(Args)
summary = quality.build_quality_summary(rows)
line_report = quality.build_quality_by_line(
    rows,
    min_observations=MIN_OBSERVATIONS,
    limit=LIMIT,
)
examples = quality.build_examples(rows, LIMIT)
summary


In [ ]:
line_report


In [ ]:
examples


In [ ]:
chart_data = summary.filter(pl.col("quality_check") != "analysis_rows")
if chart_data.is_empty():
    print("No quality flags found.")
else:
    fig = px.bar(
        chart_data,
        x="row_count",
        y="quality_check",
        orientation="h",
        title="Quality flag row counts",
        labels={"quality_check": "Quality check", "row_count": "Rows"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
